# uuid_utils smoke test

Drop-in faster UUID functions for Python, written in Rust (PyO3 / maturin).

In [ ]:
# Example 1: uuid4 (random) — the most common UUID variant.
import uuid_utils

u = uuid_utils.uuid4()
print(f"uuid4: {u}")
print(f"type: {type(u).__name__}")
print(f"version: {u.version}")
print(f"variant: {u.variant}")
print(f"hex: {u.hex}")
print(f"int: {u.int}")
print(f"bytes: {u.bytes!r}")

assert u.version == 4
assert len(str(u)) == 36
assert len(u.hex) == 32

In [ ]:
# Example 2: uuid1 (timestamp + MAC) and uuid3/uuid5 (name-based).
import uuid_utils

u1 = uuid_utils.uuid1()
print(f"uuid1: {u1} (version={u1.version})")

namespace = uuid_utils.NAMESPACE_DNS
u3 = uuid_utils.uuid3(namespace, "example.com")
u5 = uuid_utils.uuid5(namespace, "example.com")
print(f"uuid3(DNS, example.com): {u3} (version={u3.version})")
print(f"uuid5(DNS, example.com): {u5} (version={u5.version})")

# Name-based UUIDs are deterministic.
u3_again = uuid_utils.uuid3(namespace, "example.com")
u5_again = uuid_utils.uuid5(namespace, "example.com")
assert u1.version == 1
assert u3.version == 3 and u3 == u3_again
assert u5.version == 5 and u5 == u5_again

In [ ]:
# Example 3: uuid6 / uuid7 / uuid8 — the newer time-ordered variants.
import time

import uuid_utils

u6 = uuid_utils.uuid6()
u7_a = uuid_utils.uuid7()
time.sleep(0.005)
u7_b = uuid_utils.uuid7()

print(f"uuid6: {u6} (version={u6.version})")
print(f"uuid7 #1: {u7_a} (version={u7_a.version})")
print(f"uuid7 #2: {u7_b} (version={u7_b.version})")

assert u6.version == 6
assert u7_a.version == 7 and u7_b.version == 7
# uuid7 is monotonically increasing (timestamp-prefixed).
assert u7_b > u7_a, f"expected {u7_b} > {u7_a}"

In [ ]:
# Example 4: Round-trip parsing — string, hex, int, bytes constructors.
import uuid_utils

original = uuid_utils.uuid4()
s = str(original)
h = original.hex
i = original.int
b = original.bytes

from_str = uuid_utils.UUID(s)
from_hex = uuid_utils.UUID(hex=h)
from_int = uuid_utils.UUID(int=i)
from_bytes = uuid_utils.UUID(bytes=b)

print(f"original  : {original}")
print(f"from str  : {from_str}")
print(f"from hex  : {from_hex}")
print(f"from int  : {from_int}")
print(f"from bytes: {from_bytes}")

assert from_str == original
assert from_hex == original
assert from_int == original
assert from_bytes == original

In [ ]:
# Example 5: Bulk generation — uniqueness + perf sanity check.
import time

import uuid_utils

N = 50_000
start = time.perf_counter()
ids = [uuid_utils.uuid4() for _ in range(N)]
elapsed = time.perf_counter() - start

unique = {str(u) for u in ids}
print(f"generated {N} uuid4s in {elapsed:.3f}s ({N / elapsed:,.0f}/s)")
print(f"unique: {len(unique)} / {N}")

# uuid7 should generate sorted ids in insertion order.
seq = [uuid_utils.uuid7() for _ in range(1000)]
assert seq == sorted(seq), "uuid7 should be monotonically increasing"
assert len(unique) == N, f"collision detected: {N - len(unique)} duplicates"